# Qwen3-8B controlled ablations — restart v1
For the verified RTX 3060 environment. Run in a fresh kernel. No installs or model downloads are performed. Requires numpy, pandas, scipy, scikit-learn plus the packages already used in the smoke test.

**Scope:** controlled pilot ablations on the saved 57/12/13 splits. The test has already been inspected. Five training seeds do not substitute for more independent examples. Do not describe these data as a new untouched confirmatory test.

All five trained methods use label-only loss, rank 32, alpha 64, q/v/o projections, batch 1, accumulation 16, 12 epochs, fixed final checkpoint. Validation loss is logged but does not select checkpoints. Full-stack LoRA has a larger parameter budget; the other methods each use four layers. Random layers change with seed, so their variance includes placement and training randomness.

Probes and patching run on a fresh **unadapted** model. Probe features use actual block outputs, not misindexed hidden-state tensors. Patching copies block outputs before final RMSNorm, including the last block. Final-position patching is a restricted intervention; large final-block restoration alone does not demonstrate uniquely localized sentiment computation.

References: https://huggingface.co/docs/peft/developer_guides/quantization


In [ ]:
from pathlib import Path
import os, json, random, gc, time, hashlib, re, platform
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
from scipy.stats import binomtest
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, default_data_collator
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from datasets import Dataset
MODEL_PATH=Path(os.environ.get('QWEN3_MODEL_PATH', './models/qwen3-8b')).expanduser()
OUT=Path('controlled_ablation_v1') # New directory; historical outputs are untouched.
SEEDS=[42,52,62,72,82]
MAX_LENGTH=128
EPOCHS=12
K=4
OUT.mkdir(exist_ok=True)
assert torch.cuda.is_available() and torch.cuda.device_count()==1
assert MODEL_PATH.is_dir()
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
def write_json(path,obj):
    path=Path(path); tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2)); tmp.replace(path)
def digest(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def cleanup():
    gc.collect(); torch.cuda.empty_cache()
env={'python':platform.python_version(),'torch':torch.__version__,'cuda':torch.version.cuda,
 'gpu':torch.cuda.get_device_name(0),'packages':{p:version(p) for p in ['transformers','peft','bitsandbytes','accelerate','datasets','scikit-learn','scipy']}}
print(json.dumps(env,indent=2))


In [ ]:
SPLITS={'train': [{'text': 'زیبا و خوش دسته !', 'label': '1'}, {'text': 'نسبت به امکاناتش به صرفه نیست ..', 'label': '0'}, {'text': 'اصلا طرح هاشون متنوع نیست!', 'label': '0'}, {'text': 'گرمای زیاد احتمال سوختن صفحه نمایش', 'label': '0'}, {'text': 'من الان ۱ ساله که دارمش و واقعاً مشکلی\u200c ازش ندیدم.', 'label': '1'}, {'text': 'من راضی نیستم چون با اینترنتش راحت نمیشه کار کرد', 'label': '0'}, {'text': 'از برتری های اصلی این تبلت حساب میشه.', 'label': '1'}, {'text': 'یکی\u200c از بی\u200c\u200cنقص\u200cترین گوشی\u200cهایی\u200c که تا حالا داشتم.', 'label': '1'}, {'text': 'معمولیشم داشتم ولی خیلی داغ میکرد.', 'label': '0'}, {'text': 'البته زودم شارژ نمیشه چون شارژرش فست نیست', 'label': '0'}, {'text': 'و دیگه اینکه لوازم جانبیش خیلی گرونه.', 'label': '0'}, {'text': 'من بیشتر از کاربردهای زیادی که داره خوشحال هستم.', 'label': '1'}, {'text': 'صفحه نمایش خوبی هم داره.', 'label': '1'}, {'text': 'گوشی با بسته بندی عالی بدستم رسید', 'label': '1'}, {'text': 'به شدت برا دانلود ضعیف عمل می کنه', 'label': '0'}, {'text': 'به نظر چیز خوبی میاد ..', 'label': '1'}, {'text': 'ای کاش NFCهم داشت', 'label': '0'}, {'text': 'تنها عیبی که داره ..', 'label': '0'}, {'text': 'و نقش خلاقیت و ایده\u200cهای مبتکرانه کلیدی\u200cتر شده است.', 'label': '1'}, {'text': 'جدا از تخفیفاته خوبه دیجی کالا تی وی حرف نداره', 'label': '1'}, {'text': 'تا به اینجا همه چیز خوب است.', 'label': '1'}, {'text': 'طوری که اگه یه بار از دستتون بیوفته خراب می\u200cشه.', 'label': '0'}, {'text': 'نداشتن دوربین دوم', 'label': '0'}, {'text': 'به جرأت میشه گفت یه غول به تمام معنا .', 'label': '1'}, {'text': 'سلام گوشی خوب و مناسبی است اما هندزفری نداره.', 'label': '1'}, {'text': 'در کل به قیمتش می ارزه', 'label': '1'}, {'text': 'علاوه بر گارانتی یک بیمه یک سال سرقت هم داره', 'label': '1'}, {'text': 'علی الخصوص اینکه لبه\u200cهای گوشی نسل جدید دیگه خمیده نیست', 'label': '1'}, {'text': 'لرزش گیر خوبی داره واسه فیلم برداری واقعا.', 'label': '1'}, {'text': 'نداشتن رابط یو اس بی هست .', 'label': '0'}, {'text': 'البته کابل HDMI برای اتصال به HDTV ندارد.', 'label': '0'}, {'text': 'صفحه به این بزرگی دیگه گوشی نیست.', 'label': '0'}, {'text': 'من خودم این گوشی رو دارم 100 درصد راضی هستم.', 'label': '1'}, {'text': 'خلاصه دوربین جالبیه', 'label': '1'}, {'text': 'این طراحی دو ایراد را برای دوربین به وجود می\u200cآورد.', 'label': '0'}, {'text': 'ز لحاظ استحکام و قدرت باطری وتداخل هوش مصنوعی با رابط کاربری گوشی در مورد این مدل اگه نظر بدم خیلی از دوستان گرفتن راضی هستن، ولی برای من راضی کننده نبود', 'label': '0'}, {'text': 'یه اداپورت 33 واتی و شارژرم تو کارتون داره 50 تا 100 درصد رو برام تویه 1 ساعت شارژ میکنه', 'label': '1'}, {'text': 'فقط چیزایی ک این گوشی کم داره اینه ک 5G نیست', 'label': '0'}, {'text': 'عالیه اما اگر دوربینش بالاتر بود بهتر می شد', 'label': '1'}, {'text': 'من عاشق این گوشیم نقص نداره', 'label': '1'}, {'text': 'بیرون بودن دوربین هارو نمیشه ایراد گرفت اما بهتر میشد اگه در این حد بیرون نزده بود', 'label': '0'}, {'text': 'در ایزو 800 اوضاع بدتر می\u200cگردد.', 'label': '0'}, {'text': 'حتما این گوشی رو به دوستان معرفی خواهم کرد.', 'label': '1'}, {'text': 'نسبت به امکاناتش ارزش نداره', 'label': '0'}, {'text': 'همه برنامه ها و هم راحت نصب کردم', 'label': '1'}, {'text': 'اندازه این گوشی زیادی بزرگه.', 'label': '0'}, {'text': 'من که خریدم ازش راضی ام', 'label': '1'}, {'text': 'چیزی وجود نداره که بخواهید و در این دستگاه نباشه.', 'label': '1'}, {'text': 'از همه لحاظ تبلت خوبیه نسبت به قیمتش.', 'label': '1'}, {'text': 'به نظر من با این قیمتی که داره ارزش خرید نداره', 'label': '0'}, {'text': 'فریم که میشد از آلومینیوم باشه', 'label': '0'}, {'text': 'برا داخل جیب گذاشتن بزرگه', 'label': '0'}, {'text': 'کالاف دیوتی موبایل روی فریم ریت مکس بازی می\u200cکنم بدون داغ شدن گوشی یا لگ و باگی', 'label': '1'}, {'text': 'کاور مخصوص داخلش نبود', 'label': '0'}, {'text': 'خیلی روون و نرمه صفحه نمایشش', 'label': '1'}, {'text': 'باوجود باطری 6500 زود خالی میشه', 'label': '0'}, {'text': 'نمیدونم سامسونگ به جای دادن i9100g که ضعیف هستش.', 'label': '0'}], 'validation': [{'text': 'قاب دورش هم خیلی باریکه.', 'label': '1'}, {'text': 'دوربین خیلی ضعیفه اصلا روش حساب خاصی باز نکنید', 'label': '0'}, {'text': 'از نظر ظاهری هم خوبه', 'label': '1'}, {'text': 'به نظر من دو مشکل اساسی داره 1.', 'label': '0'}, {'text': 'باید نور رو تا نزدیکای آخر ببری که نور خوب بده', 'label': '0'}, {'text': 'نه خیلی سنگین و نه خیلی سبک.', 'label': '1'}, {'text': 'من یک سال پیش خریدمش فقط به خاطر داغ\u200cکردنش فروختمش', 'label': '0'}, {'text': 'در مورد دوربین اصلا چیزی نگم بهتره واقعا بده دوربینش راضی کننده نیست', 'label': '0'}, {'text': 'نسبت به پولش واقعا کارایی داره.', 'label': '1'}, {'text': 'طراحی زیبایی هم نداره!', 'label': '0'}, {'text': 'ظاهرش هم به نظر من از SIII بهتره!', 'label': '1'}, {'text': 'همه چیزش هم در حد خوبیه.', 'label': '1'}], 'test': [{'text': 'از لحاظ باتری که نسبت به هم رده هاش سره.', 'label': '1'}, {'text': 'کیفتش super amoled plus نیست.', 'label': '0'}, {'text': 'من جطوری ماینکرفت بریزم هی میگه با گوشی سازگار نیست', 'label': '0'}, {'text': 'اینترنتش قطع و وصل میشه', 'label': '0'}, {'text': 'باتری مناسب و خوب برای تبلت 12.', 'label': '1'}, {'text': 'تلویزیون خوبیه نسبت به قیمتش.', 'label': '1'}, {'text': 'به نظر من هیچ فرقی با نوت2 نداره.', 'label': '0'}, {'text': 'قابلیت های ai برام جالبه؛', 'label': '1'}, {'text': 'دیگه هیچی کم نداره از دستش ندین دوستان', 'label': '1'}, {'text': 'این گوشی nfcو 4G نداره.', 'label': '0'}, {'text': 'به نظر من این s pen اصلا کاربرد نداره!', 'label': '0'}, {'text': 'خودش قطع کن شارژ داره', 'label': '1'}, {'text': 'کاش full HD بود', 'label': '0'}]}
PATCH_PAIRS=[{'clean_text': 'Text: کیفیت دوربین این گوشی خوب است. -> Label: ', 'corrupt_text': 'Text: کیفیت دوربین این گوشی خراب است. -> Label: ', 'target_label': '1', 'pair_id': '1', 'phenomenon': 'lexical_polarity', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: باتری دستگاه ماندگاری بالایی دارد. -> Label: ', 'corrupt_text': 'Text: باتری دستگاه ماندگاری کمی دارد. -> Label: ', 'target_label': '1', 'pair_id': '2', 'phenomenon': 'battery', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: صفحه\u200cنمایش روشن و خوانا است. -> Label: ', 'corrupt_text': 'Text: صفحه\u200cنمایش محو و تیره است. -> Label: ', 'target_label': '1', 'pair_id': '3', 'phenomenon': 'display', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: صدای بلندگو کامل واضح است. -> Label: ', 'corrupt_text': 'Text: صدای بلندگو غیر واضح است. -> Label: ', 'target_label': '1', 'pair_id': '4', 'phenomenon': 'audio', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: بدنه گوشی هم محکم و هم بادوام است. -> Label: ', 'corrupt_text': 'Text: بدنه گوشی هم ضعیف و کم\u200cدوام است. -> Label: ', 'target_label': '1', 'pair_id': '5', 'phenomenon': 'build_quality', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: اجرای برنامه\u200cها سریع و روان است. -> Label: ', 'corrupt_text': 'Text: اجرای برنامه\u200cهاش کمی سنگینه. -> Label: ', 'target_label': '1', 'pair_id': '6', 'phenomenon': 'performance', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: رابط کاربری ساده و روان است. -> Label: ', 'corrupt_text': 'Text: رابط کاربری دشوار و کند است. -> Label: ', 'target_label': '1', 'pair_id': '7', 'phenomenon': 'usability', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: این محصول نسبت به امکاناتش به\u200cصرفه است. -> Label: ', 'corrupt_text': 'Text: این محصول نسبت به امکانات خب گران است. -> Label: ', 'target_label': '1', 'pair_id': '8', 'phenomenon': 'value', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: طراحی گوشی زیبا و جذاب است. -> Label: ', 'corrupt_text': 'Text: طراحی گوشی زشت و قدیمی است. -> Label: ', 'target_label': '1', 'pair_id': '9', 'phenomenon': 'design', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: شارژر دستگاه به\u200cخوبی کار می\u200cکند. -> Label: ', 'corrupt_text': 'Text: شارژر دستگاه به\u200cسختی کار می\u200cکند. -> Label: ', 'target_label': '1', 'pair_id': '10', 'phenomenon': 'accessory', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: آنتن\u200cدهی گوشی خوب و پایدار است. -> Label: ', 'corrupt_text': 'Text: آنتن\u200cدهی گوشی ضعیف و بد است. -> Label: ', 'target_label': '1', 'pair_id': '11', 'phenomenon': 'connectivity', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: حافظه داخلی برای استفاده روزمره جوابگو است. -> Label: ', 'corrupt_text': 'Text: حافظه داخلی برای استفاده روزمره جواب نیست. -> Label: ', 'target_label': '1', 'pair_id': '12', 'phenomenon': 'storage', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: حسگر اثر انگشت سریع و دقیق است. -> Label: ', 'corrupt_text': 'Text: حسگر اثر انگشت کند و نادقیق است. -> Label: ', 'target_label': '1', 'pair_id': '13', 'phenomenon': 'sensor', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: وزن گوشی سبک و مناسب است. -> Label: ', 'corrupt_text': 'Text: وزن گوشی کمی سنگینه. -> Label: ', 'target_label': '1', 'pair_id': '14', 'phenomenon': 'ergonomics', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: به\u200cروزرسانی\u200cهای نرم\u200cافزاری به\u200cموقع منتشر می\u200cشوند. -> Label: ', 'corrupt_text': 'Text: به\u200cروزرسانی\u200cهای نرم\u200cافزاری باتأخیر منتشر می شدند. -> Label: ', 'target_label': '1', 'pair_id': '15', 'phenomenon': 'software', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: از خرید این گوشی کاملاً راضی\u200cام. -> Label: ', 'corrupt_text': 'Text: از خرید این گوشی کاملاً ناراضیم. -> Label: ', 'target_label': '1', 'pair_id': '16', 'phenomenon': 'overall_satisfaction', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: عکس\u200cهای شبانه جزئیات خوبی دارند. -> Label: ', 'corrupt_text': 'Text: عکس\u200cهای شبانه جزئیات کمی دارند. -> Label: ', 'target_label': '1', 'pair_id': '17', 'phenomenon': 'camera', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: با وجود قیمت بالا، کیفیت دستگاه مطلوب است. -> Label: ', 'corrupt_text': 'Text: با وجود قیمت بالا، کیفیت دستگاه پایینه. -> Label: ', 'target_label': '1', 'pair_id': '18', 'phenomenon': 'contrast', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: ابتدا تردید داشتم، اما از عملکردش راضی هستم. -> Label: ', 'corrupt_text': 'Text: ابتدا تردید داشتم، اما از عملکردش ناراضی\u200cام. -> Label: ', 'target_label': '1', 'pair_id': '19', 'phenomenon': 'contrast', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: این گوشی واقعاً خیلی خوبه. -> Label: ', 'corrupt_text': 'Text: این گوشی درواقع خیلی افتضاح است. -> Label: ', 'target_label': '1', 'pair_id': '20', 'phenomenon': 'colloquial', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: کیفیت عکس\u200cهای این گوشی پایین است. -> Label: ', 'corrupt_text': 'Text: کیفیت عکس\u200cهای این گوشی عالی است. -> Label: ', 'target_label': '0', 'pair_id': '31', 'phenomenon': 'camera', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: شارژدهی باتری اعصاب خردکن است. -> Label: ', 'corrupt_text': 'Text: شارژدهی باتری رضایت\u200cبخش است. -> Label: ', 'target_label': '0', 'pair_id': '32', 'phenomenon': 'battery', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: رنگ\u200cهای صفحه\u200cنمایش غیرزنده و بی\u200cروح هستند. -> Label: ', 'corrupt_text': 'Text: رنگ\u200cهای صفحه\u200cنمایش زنده و طبیعی هستند. -> Label: ', 'target_label': '0', 'pair_id': '33', 'phenomenon': 'display', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: صدای مکالمه نامفهوم است. -> Label: ', 'corrupt_text': 'Text: صدای مکالمه واضح و قوی است. -> Label: ', 'target_label': '0', 'pair_id': '34', 'phenomenon': 'audio', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: بدنه دستگاه لق و ضعیف است. -> Label: ', 'corrupt_text': 'Text: بدنه دستگاه قوی و مقاوم است. -> Label: ', 'target_label': '0', 'pair_id': '35', 'phenomenon': 'build_quality', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: برنامه\u200cها روی این گوشی زور اجرا می\u200cشوند. -> Label: ', 'corrupt_text': 'Text: برنامه\u200cها روی این گوشی راحت اجرا می\u200cشوند. -> Label: ', 'target_label': '0', 'pair_id': '36', 'phenomenon': 'performance', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: منوهای دستگاه غیرواضح هستند. -> Label: ', 'corrupt_text': 'Text: منوهای دستگاه ساده و واضح هستند. -> Label: ', 'target_label': '0', 'pair_id': '37', 'phenomenon': 'usability', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: قیمت این مدل با این امکانات خوب نیست. -> Label: ', 'corrupt_text': 'Text: قیمت این مدل به ازای امکاناتش مناسب است. -> Label: ', 'target_label': '0', 'pair_id': '38', 'phenomenon': 'value', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: ظاهر گوشی از مد افتاده است. -> Label: ', 'corrupt_text': 'Text: ظاهر گوشی قشنگ و جدید است. -> Label: ', 'target_label': '0', 'pair_id': '39', 'phenomenon': 'design', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: کابل همراه دستگاه  بی کیفیت است. -> Label: ', 'corrupt_text': 'Text: کابل همراه دستگاه با کیفیت خوب است. -> Label: ', 'target_label': '0', 'pair_id': '40', 'phenomenon': 'accessory', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: آنتن\u200cدهی در محیط بسته پایین است. -> Label: ', 'corrupt_text': 'Text: آنتن\u200cدهی در محیط بسته قوی است. -> Label: ', 'target_label': '0', 'pair_id': '41', 'phenomenon': 'connectivity', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: فضای ذخیره\u200cسازی برای برنامه\u200cها کم است. -> Label: ', 'corrupt_text': 'Text: فضای ذخیره\u200cسازی برای برنامه\u200cها بس است. -> Label: ', 'target_label': '0', 'pair_id': '42', 'phenomenon': 'storage', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: حسگر اثر انگشت خطای بالایی دارد. -> Label: ', 'corrupt_text': 'Text: حسگر اثر انگشت دقت بالایی دارد. -> Label: ', 'target_label': '0', 'pair_id': '43', 'phenomenon': 'sensor', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: از انتخاب این مدل کاملاً پشیمانم. -> Label: ', 'corrupt_text': 'Text: از انتخاب این مدل کاملاً راضی\u200cام. -> Label: ', 'target_label': '0', 'pair_id': '46', 'phenomenon': 'overall_satisfaction', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: بعد از به\u200cروزرسانی، عملکرد گوشی بدتر شد. -> Label: ', 'corrupt_text': 'Text: بعد از به\u200cروزرسانی، عملکرد گوشی بهتر شد. -> Label: ', 'target_label': '0', 'pair_id': '49', 'phenomenon': 'software', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: گوشی هنگام بازی خیلی داغ می\u200cکند. -> Label: ', 'corrupt_text': 'Text: گوشی هنگام بازی اصلاً داغ نمی\u200cکند. -> Label: ', 'target_label': '0', 'pair_id': '53', 'phenomenon': 'thermal', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: کیفیت این مدل کمتر از انتظارم بود. -> Label: ', 'corrupt_text': 'Text: کیفیت این مدل بیش از انتظارم بود. -> Label: ', 'target_label': '0', 'pair_id': '56', 'phenomenon': 'expectation', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: خرید این گوشی را توصیه نمی\u200cکنم. -> Label: ', 'corrupt_text': 'Text: خرید این گوشیو توصیه می\u200cکنیم. -> Label: ', 'target_label': '0', 'pair_id': '57', 'phenomenon': 'recommendation', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: این محصول ارزش خرید ندارد. -> Label: ', 'corrupt_text': 'Text: این محصول ارزش خریدن دارد. -> Label: ', 'target_label': '0', 'pair_id': '58', 'phenomenon': 'value', 'source': 'synthetic_minimal_pair'}, {'clean_text': 'Text: در استفاده روزانه مشکلات زیادی داشتم. -> Label: ', 'corrupt_text': 'Text: در استفاده روزانه مشکل خاصی نداشتم. -> Label: ', 'target_label': '0', 'pair_id': '59', 'phenomenon': 'reliability', 'source': 'synthetic_minimal_pair'}]
frames={k:pd.DataFrame(v).astype({'label':int}) for k,v in SPLITS.items()}
train_df,val_df,test_df=[frames[k] for k in ['train','validation','test']]
def norm(t): return re.sub(r'\s+',' ',t.replace('ي','ی').replace('ك','ک')).strip()
sets={k:set(f.text.map(norm)) for k,f in frames.items()}
for a,b in [('train','validation'),('train','test'),('validation','test')]:
    assert not sets[a]&sets[b], f'Overlap between {a} and {b}'
for k,f in frames.items():
    assert not f.text.map(norm).duplicated().any(), f'Normalized duplicates in {k}'
    assert set(f.label)=={0,1}
    print(k,len(f),f.label.value_counts().to_dict())
protocol={'model_path':str(MODEL_PATH),'config_sha256':hashlib.sha256((MODEL_PATH/'config.json').read_bytes()).hexdigest(),
 'split_hash':digest(SPLITS),'pair_hash':digest(PATCH_PAIRS),'seeds':SEEDS,'max_length':MAX_LENGTH,
 'epochs':EPOCHS,'batch':1,'accumulation':16,'rank':32,'alpha':64,'lr':2e-4,
 'modules':['q_proj','v_proj','o_proj'],'objective':'label only, no EOS','checkpoint':'fixed final',
 'selection':'probe top3 then patch top unique until four','environment':env}
protocol_hash=digest(protocol)
if (OUT/'protocol.json').exists():
    assert json.loads((OUT/'protocol.json').read_text())==protocol,'Protocol changed: choose a new OUT directory'
else: write_json(OUT/'protocol.json',protocol)
tokenizer=AutoTokenizer.from_pretrained(str(MODEL_PATH),local_files_only=True)
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
label_ids={y:tokenizer.encode(str(y),add_special_tokens=False) for y in [0,1]}
assert all(len(v)==1 for v in label_ids.values())
HEAD=tokenizer.encode('Text: ',add_special_tokens=False)
SUFFIX=tokenizer.encode(' -> Label: ',add_special_tokens=False)
def prefix_ids(text):
    budget=MAX_LENGTH-len(HEAD)-len(SUFFIX)-1
    assert budget>0
    return HEAD+tokenizer.encode(text,add_special_tokens=False)[:budget]+SUFFIX
def encode(row):
    prefix=prefix_ids(row['text']); target=label_ids[int(row['label'])]
    ids=prefix+target; labels=[-100]*len(prefix)+target; mask=[1]*len(ids)
    pad=MAX_LENGTH-len(ids)
    ids += [tokenizer.pad_token_id]*pad; labels += [-100]*pad; mask += [0]*pad
    assert sum(v!=-100 for v in labels)==1
    return dict(input_ids=ids,labels=labels,attention_mask=mask)
encode({'text':'کیفیت محصول '*1000,'label':1})
train_tok=Dataset.from_list([encode(r) for r in train_df.to_dict('records')])
val_tok=Dataset.from_list([encode(r) for r in val_df.to_dict('records')])
def load_base(training=False):
    m=AutoModelForCausalLM.from_pretrained(str(MODEL_PATH),local_files_only=True,
       quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
       bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16),
       device_map={'':0},torch_dtype=torch.float16,attn_implementation='sdpa')
    assert m.config.num_hidden_layers==36
    assert not any('lora_' in n for n,_ in m.named_parameters())
    m.config.use_cache=False
    m=prepare_model_for_kbit_training(m,use_gradient_checkpointing=training,gradient_checkpointing_kwargs={'use_reentrant':False})
    m.eval(); return m
def add_adapter(m,layers):
    return get_peft_model(m,LoraConfig(r=32,lora_alpha=64,lora_dropout=0.05,bias='none',
      target_modules=['q_proj','v_proj','o_proj'],layers_to_transform=layers,task_type='CAUSAL_LM'))
def training_args(path,seed,smoke=False):
    return TrainingArguments(output_dir=str(path),per_device_train_batch_size=1,
      gradient_accumulation_steps=16,num_train_epochs=EPOCHS,max_steps=2 if smoke else -1,
      learning_rate=2e-4,lr_scheduler_type='cosine',fp16=True,bf16=False,
      optim='paged_adamw_8bit',max_grad_norm=0.3,gradient_checkpointing=True,
      gradient_checkpointing_kwargs={'use_reentrant':False},eval_strategy='no' if smoke else 'epoch',
      per_device_eval_batch_size=1,save_strategy='no',logging_strategy='steps',logging_steps=5,
      report_to=[],seed=seed,data_seed=seed,dataloader_num_workers=0)


## 1. Full-stack memory preflight
This is a separate two-step run with all 36 layers. If it fails with out-of-memory, stop and send the traceback. Do not silently reduce rank for one method; that would change the comparison. Restart the kernel before retrying.


In [ ]:
if not (OUT/'full_stack_preflight.json').exists():
    seed_all(42)
    m=add_adapter(load_base(training=True),list(range(36)))
    t=Trainer(model=m,args=training_args(OUT/'preflight',42,True),train_dataset=train_tok.select(range(4)),data_collator=default_data_collator)
    torch.cuda.reset_peak_memory_stats(); result=t.train()
    assert t.state.global_step==2 and np.isfinite(result.training_loss)
    write_json(OUT/'full_stack_preflight.json',{'passed':True,'training_loss':result.training_loss,
      'peak_allocated_gib':torch.cuda.max_memory_allocated()/2**30,'peak_reserved_gib':torch.cuda.max_memory_reserved()/2**30})
    del t,m;cleanup()
print(json.loads((OUT/'full_stack_preflight.json').read_text()))


## 2. Fresh-backbone layer selection
Probing uses training-only five-fold CV with scaling fitted inside each fold. This small-data ranking is a selection heuristic, not proof of mechanistic importance. Synthetic patch pairs never enter adapter training or evaluation. Both clean and corrupt strings are already complete prompts.


In [ ]:
def inputs_from_ids(ids):
    x=torch.tensor([ids],device='cuda:0'); return {'input_ids':x,'attention_mask':torch.ones_like(x)}
@torch.no_grad()
def margin(m,inputs):
    logits=m(**inputs,use_cache=False).logits[0,-1].float()
    return float((logits[label_ids[1][0]]-logits[label_ids[0][0]]).item())
def output_tensor(output):
    return output[0] if isinstance(output,tuple) else output
@torch.no_grad()
def block_features(m,text):
    states={}; handles=[]
    for layer,block in enumerate(m.model.layers):
        def hook(mod,args,output,layer=layer):
            states[layer]=output_tensor(output)[0,-1].float().cpu().numpy()
        handles.append(block.register_forward_hook(hook))
    try: m(**inputs_from_ids(prefix_ids(text)),use_cache=False)
    finally:
        for h in handles:h.remove()
    return np.stack([states[i] for i in range(36)])
@torch.no_grad()
def patch_pair(m,row,pair_id):
    clean_ids=tokenizer.encode(row['clean_text'],add_special_tokens=False)
    corrupt_ids=tokenizer.encode(row['corrupt_text'],add_special_tokens=False)
    assert len(clean_ids)==len(corrupt_ids),'Unequal token lengths; review alignment'
    assert max(len(clean_ids),len(corrupt_ids))<=MAX_LENGTH
    assert row['clean_text'].startswith('Text: ') and row['clean_text'].endswith(' -> Label: ')
    clean=inputs_from_ids(clean_ids);corrupt=inputs_from_ids(corrupt_ids)
    states={};handles=[]
    for layer,block in enumerate(m.model.layers):
        def capture(mod,args,output,layer=layer):
            states[layer]=output_tensor(output)[:,-1,:].detach().cpu()
        handles.append(block.register_forward_hook(capture))
    try: clean_margin=margin(m,clean)
    finally:
        for h in handles:h.remove()
    corrupt_margin=margin(m,corrupt)
    direction=1 if int(row['target_label'])==1 else -1
    gap=direction*(clean_margin-corrupt_margin)
    records=[]
    for layer,block in enumerate(m.model.layers):
        def replace(mod,args,output,layer=layer):
            value=output_tensor(output).clone()
            value[:,-1,:]=states[layer].to(value.device,value.dtype)
            return (value,)+output[1:] if isinstance(output,tuple) else value
        handle=block.register_forward_hook(replace)
        try: patched=margin(m,corrupt)
        finally: handle.remove()
        recovery=direction*(patched-corrupt_margin)
        records.append({'pair_id':pair_id,'layer':layer,'target_label':int(row['target_label']),
          'clean_margin':clean_margin,'corrupt_margin':corrupt_margin,'patched_margin':patched,
          'oriented_recovery':recovery,'clean_corrupt_gap':gap,
          'normalized_restoration':recovery/gap if gap>1e-6 else None})
    return records

if not (OUT/'selection.json').exists():
    seed_all(42);m=load_base()
    features=np.stack([block_features(m,text) for text in train_df.text])
    cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
    probe_rows=[]
    for layer in range(36):
        probe=make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,solver='liblinear',random_state=42))
        scores=cross_val_score(probe,features[:,layer,:],train_df.label,cv=cv,scoring='f1_macro')
        probe_rows.append({'layer':layer,'macro_f1_mean':float(scores.mean()),'macro_f1_sd':float(scores.std(ddof=1))})
    probes=pd.DataFrame(probe_rows).sort_values(['macro_f1_mean','layer'],ascending=[False,True])
    probes.to_csv(OUT/'probe_ranking.csv',index=False)
    patch_rows=[]
    for i,row in enumerate(PATCH_PAIRS):
        patch_rows.extend(patch_pair(m,row,i)); print('Patching pair',i+1,'/',len(PATCH_PAIRS))
    raw=pd.DataFrame(patch_rows);raw.to_csv(OUT/'patch_pair_results.csv',index=False)
    summary=raw.groupby('layer').oriented_recovery.agg(['mean','std','count']).reset_index().sort_values(['mean','layer'],ascending=[False,True])
    summary.to_csv(OUT/'patch_ranking.csv',index=False)
    probe_layers=probes.layer.astype(int).head(K).tolist()
    patch_layers=summary.layer.astype(int).head(K).tolist()
    combined=list(dict.fromkeys(probe_layers[:3]+summary.layer.astype(int).tolist()))[:K]
    selection={'probe_only':probe_layers,'patch_only':patch_layers,'combined':combined}
    assert all(len(v)==K and len(set(v))==K for v in selection.values())
    write_json(OUT/'selection.json',selection)
    del m,features;cleanup()
selection=json.loads((OUT/'selection.json').read_text()); print(selection)


## 3. Controlled runs
Each completed run is skipped on restart only when its configuration matches. A failure is recorded and stops execution. Metrics are committed after predictions and adapter save succeed. Expect 25 training runs plus base evaluation. Time estimates should come from the first real run, not the four-second smoke test.


In [ ]:
@torch.no_grad()
def evaluate(m,frame):
    m.eval();rows=[]
    for i,row in frame.reset_index(drop=True).iterrows():
        diff=margin(m,inputs_from_ids(prefix_ids(row.text)))
        rows.append({'example_id':i,'text':row.text,'y_true':int(row.label),'y_pred':int(diff>0),'logit_margin':diff})
    predictions=pd.DataFrame(rows)
    y=predictions.y_true;p=predictions.y_pred
    metrics={'n':len(y),'accuracy':float(accuracy_score(y,p)),
      'macro_f1':float(f1_score(y,p,average='macro',labels=[0,1],zero_division=0)),
      'mcc':float(matthews_corrcoef(y,p)),'confusion_matrix':confusion_matrix(y,p,labels=[0,1]).tolist()}
    return metrics,predictions
if not (OUT/'base_metrics.json').exists():
    m=load_base();metrics,pred=evaluate(m,test_df)
    pred.to_csv(OUT/'base_predictions.csv',index=False)
    write_json(OUT/'base_metrics.json',metrics)
    del m;cleanup()

def run_one(method,layers,seed):
    run_dir=OUT/'runs'/method/str(seed);run_dir.mkdir(parents=True,exist_ok=True)
    config={'protocol_hash':protocol_hash,'method':method,'seed':seed,'layers':layers}
    if (run_dir/'config.json').exists():
        assert json.loads((run_dir/'config.json').read_text())==config,'Run configuration mismatch'
    else:write_json(run_dir/'config.json',config)
    if (run_dir/'metrics.json').exists():
        assert (run_dir/'predictions.csv').exists() and (run_dir/'adapter'/'adapter_config.json').exists()
        print('Skipping completed',method,seed);return
    seed_all(seed);m=None;t=None
    try:
        m=add_adapter(load_base(training=True),layers)
        count=sum(p.numel() for p in m.parameters() if p.requires_grad)
        t=Trainer(model=m,args=training_args(run_dir/'trainer',seed),train_dataset=train_tok,
          eval_dataset=val_tok,data_collator=default_data_collator)
        torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats();start=time.perf_counter()
        result=t.train();torch.cuda.synchronize();seconds=time.perf_counter()-start
        allocated=torch.cuda.max_memory_allocated()/2**30;reserved=torch.cuda.max_memory_reserved()/2**30
        assert np.isfinite(result.training_loss)
        validation,vpred=evaluate(m,val_df)
        metrics,pred=evaluate(m,test_df)
        metrics.update(config);metrics.update(training_seconds=seconds,peak_allocated_gib=allocated,
          peak_reserved_gib=reserved,trainable_parameters=count,training_loss=float(result.training_loss),validation=validation)
        pred.to_csv(run_dir/'predictions.csv',index=False)
        vpred.to_csv(run_dir/'validation_predictions.csv',index=False)
        write_json(run_dir/'training_log.json',t.state.log_history)
        m.save_pretrained(run_dir/'adapter');tokenizer.save_pretrained(run_dir/'adapter')
        write_json(run_dir/'metrics.json',metrics)
        print(method,seed,'macro-F1',metrics['macro_f1'],'seconds',round(seconds))
    except Exception as e:
        write_json(run_dir/'failure.json',{'type':type(e).__name__,'message':str(e),'config':config})
        raise
    finally:
        del t,m;cleanup()

for seed in SEEDS:
    methods={'full_stack':list(range(36)),**selection,
      'random_four':sorted(np.random.default_rng(seed).choice(36,K,replace=False).astype(int).tolist())}
    for method,layers in methods.items():run_one(method,layers,seed)


## 4. Pilot reporting
Mean/SD measure variation across training seeds on the same test. Paired bootstrap intervals resample the 13 test examples within each seed. Seeds are not pooled as independent test observations. Exact McNemar tests assess accuracy, not macro-F1; Holm correction is applied across the reported comparisons. Small test size limits inference regardless of seed count.


In [ ]:
methods=['full_stack','random_four','probe_only','patch_only','combined']
rows=[]
for method in methods:
    for seed in SEEDS:
        path=OUT/'runs'/method/str(seed)/'metrics.json'
        assert path.exists(),f'Missing run: {method}/{seed}'
        rows.append(json.loads(path.read_text()))
metrics=pd.DataFrame(rows);metrics.to_json(OUT/'seed_metrics.json',orient='records',indent=2)
cols=['accuracy','macro_f1','mcc','training_seconds','peak_allocated_gib','trainable_parameters']
summary=metrics.groupby('method')[cols].agg(['mean','std'])
summary.columns=['_'.join(c) for c in summary.columns];summary.to_csv(OUT/'aggregate.csv')
base=pd.read_csv(OUT/'base_predictions.csv')
comparisons=[]
for seed in SEEDS:
    a=pd.read_csv(OUT/'runs'/'combined'/str(seed)/'predictions.csv').sort_values('example_id')
    for method in ['base','full_stack','random_four','probe_only','patch_only']:
        b=base if method=='base' else pd.read_csv(OUT/'runs'/method/str(seed)/'predictions.csv')
        b=b.sort_values('example_id')
        assert a[['example_id','text','y_true']].equals(b[['example_id','text','y_true']])
        y=a.y_true.to_numpy();pa=a.y_pred.to_numpy();pb=b.y_pred.to_numpy()
        ca=pa==y;cb=pb==y
        wins=int((ca&~cb).sum());losses=int((~ca&cb).sum());discordant=wins+losses
        p=binomtest(wins,discordant,0.5).pvalue if discordant else 1.0
        rng=np.random.default_rng(seed);diffs=[]
        for _ in range(2000):
            idx=rng.integers(0,len(y),len(y))
            diffs.append(f1_score(y[idx],pa[idx],labels=[0,1],average='macro',zero_division=0)-f1_score(y[idx],pb[idx],labels=[0,1],average='macro',zero_division=0))
        low,high=np.quantile(diffs,[.025,.975])
        comparisons.append({'seed':seed,'comparison':'combined_vs_'+method,
          'macro_f1_difference':float(f1_score(y,pa,average='macro')-f1_score(y,pb,average='macro')),
          'bootstrap_95_low':float(low),'bootstrap_95_high':float(high),
          'combined_only_correct':wins,'baseline_only_correct':losses,'mcnemar_p':float(p)})
order=sorted(range(len(comparisons)),key=lambda i:comparisons[i]['mcnemar_p'])
running=0
for rank,i in enumerate(order):
    running=max(running,min(1,(len(order)-rank)*comparisons[i]['mcnemar_p']))
    comparisons[i]['mcnemar_holm_p']=running
pd.DataFrame(comparisons).to_csv(OUT/'paired_comparisons.csv',index=False)
print(summary.to_string())
print('Complete. Send aggregate.csv, seed_metrics.json, selection.json and paired_comparisons.csv.')


## Submission work still required
Audit pair semantics and label quality, examine near-duplicate/product grouping, enlarge evaluation data or design nested CV, test layer-selection stability and additional random placements, add probe permutation controls and patch pair uncertainty, and benchmark generalization on an independent corpus. The manuscript's earlier five-sentence demonstration and tiny full-fine-tuning speed comparison should not be used as headline evidence.

This notebook is a corrected pilot experiment implementation, not a claim that all publication requirements are satisfied. Assistant validation covers Python syntax, notebook schema, embedded split counts and selection bookkeeping; no new GPU experiments have been run by the assistant.
